# N-BaIoT Preprocessing — Step 2: Clean
**Device: Danmini Doorbell**

This notebook starts from the Step 1 labelled checkpoint. It does **not** read or extract RAR files.

### What this step does
1. Find the Step 1 checkpoint.
2. Read it in chunks to reduce memory usage.
3. Convert positive/negative infinity to missing values.
4. Remove rows with missing or non-numeric feature values.
5. Inspect duplicate rows, but **do not automatically delete duplicates**.
6. Save a cleaned checkpoint for Step 3.

**Important:** The `label` column is kept unchanged. Label encoding belongs in Step 3, according to our preprocessing plan.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import os

# ===== CONFIGURATION =====
DEVICE = "Danmini_Doorbell"
CHUNK_SIZE = 250_000

# Locate the folder that contains "preprocessed".
cwd = Path.cwd()
search_roots = [cwd, *cwd.parents]
BASE_DIR = None

for root in search_roots:
    if (root / "preprocessed").is_dir():
        BASE_DIR = root
        break

if BASE_DIR is None:
    # Also search below the current working directory.
    matches = [p for p in cwd.rglob("preprocessed") if p.is_dir()]
    if matches:
        BASE_DIR = matches[0].parent

if BASE_DIR is None:
    raise FileNotFoundError(
        "Could not find the 'preprocessed' folder. "
        "Run Step 1 first and make sure its output folder is available."
    )

INPUT_PATH = BASE_DIR / "preprocessed" / f"{DEVICE}_step1_raw_labelled.csv"
OUTPUT_PATH = BASE_DIR / "preprocessed" / f"{DEVICE}_step2_cleaned.csv"
REPORT_PATH = BASE_DIR / "preprocessed" / f"{DEVICE}_step2_cleaning_report.json"

if not INPUT_PATH.is_file():
    raise FileNotFoundError(
        f"Step 1 checkpoint not found: {INPUT_PATH}\n"
        "Run Step 1 successfully before running Step 2."
    )

print("Base directory :", BASE_DIR)
print("Input file     :", INPUT_PATH)
print("Output file    :", OUTPUT_PATH)
print("Chunk size     :", f"{CHUNK_SIZE:,} rows")


---
## 2.1 Inspect the input checkpoint

We first inspect the column names and a small sample. N-BaIoT is expected to contain 115 numeric traffic features plus the `label` target column.


In [ ]:
sample = pd.read_csv(INPUT_PATH, nrows=5)
print("Sample shape:", sample.shape)
print("First columns:", sample.columns[:10].tolist())
print("\nSample labels:")
print(sample["label"].value_counts() if "label" in sample.columns else "ERROR: label column missing")
display(sample.head())


In [ ]:
if "label" not in sample.columns:
    raise ValueError("The Step 1 checkpoint has no 'label' column.")

feature_columns = [c for c in sample.columns if c != "label"]
print("Feature count:", len(feature_columns))
if len(feature_columns) != 115:
    print(f"⚠️ Expected 115 features, but found {len(feature_columns)}. Check the Step 1 output.")
else:
    print("✓ Found the expected 115 feature columns.")

non_numeric = sample[feature_columns].select_dtypes(exclude=[np.number]).columns.tolist()
print("Non-numeric feature columns in sample:", non_numeric)


---
## 2.2 Clean the data in chunks

For each chunk:
- Keep `label` as it is.
- Convert feature columns to numeric values. Any value that cannot be parsed becomes missing (`NaN`).
- Replace `+inf` and `-inf` with `NaN`.
- Drop rows that contain a missing value in any feature.

We drop invalid-feature rows rather than inventing measurements through imputation. The report records how many rows are removed. If the number is unexpectedly large, stop and investigate before proceeding.


In [ ]:
# Reset any previous output so rerunning this cell does not append duplicate data.
if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()

stats = {
    "device": DEVICE,
    "input_file": str(INPUT_PATH),
    "output_file": str(OUTPUT_PATH),
    "chunk_size": CHUNK_SIZE,
    "rows_read": 0,
    "rows_written": 0,
    "rows_dropped_missing_or_non_numeric": 0,
    "positive_infinity_values": 0,
    "negative_infinity_values": 0,
    "chunks_processed": 0,
    "labels": {}
}

first_chunk = True

for chunk_number, chunk in enumerate(
    pd.read_csv(INPUT_PATH, chunksize=CHUNK_SIZE, low_memory=False),
    start=1
):
    if "label" not in chunk.columns:
        raise ValueError("A chunk is missing the 'label' column.")

    current_features = [c for c in chunk.columns if c != "label"]
    if chunk_number == 1:
        feature_columns = current_features

    stats["rows_read"] += len(chunk)
    stats["chunks_processed"] += 1

    # Count infinities before replacing them.
    numeric_chunk = chunk[feature_columns].apply(pd.to_numeric, errors="coerce")
    stats["positive_infinity_values"] += int(np.isposinf(numeric_chunk.to_numpy()).sum())
    stats["negative_infinity_values"] += int(np.isneginf(numeric_chunk.to_numpy()).sum())

    # Non-numeric values become NaN; infinities also become NaN.
    numeric_chunk = numeric_chunk.replace([np.inf, -np.inf], np.nan)

    # Keep the original labels unchanged.
    labels = chunk["label"].astype("string")
    valid_rows = numeric_chunk.notna().all(axis=1) & labels.notna() & labels.ne("")

    cleaned = numeric_chunk.loc[valid_rows].copy()
    cleaned["label"] = labels.loc[valid_rows].astype(str)

    dropped = len(chunk) - len(cleaned)
    stats["rows_dropped_missing_or_non_numeric"] += int(dropped)
    stats["rows_written"] += len(cleaned)

    counts = cleaned["label"].value_counts()
    for label, count in counts.items():
        stats["labels"][label] = stats["labels"].get(label, 0) + int(count)

    cleaned.to_csv(
        OUTPUT_PATH,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )
    first_chunk = False

    print(
        f"Chunk {chunk_number:>3}: read {len(chunk):>9,} | "
        f"kept {len(cleaned):>9,} | dropped {dropped:>7,}"
    )

if stats["rows_written"] == 0:
    raise RuntimeError("No rows remain after cleaning. Inspect the input data.")

print("\n✓ Cleaning pass complete.")
print(f"Rows read    : {stats['rows_read']:,}")
print(f"Rows kept    : {stats['rows_written']:,}")
print(f"Rows dropped : {stats['rows_dropped_missing_or_non_numeric']:,}")


---
## 2.3 Review cleaning results

Check the number of rows removed and the class distribution. A class disappearing or losing a very large fraction of its rows should be investigated before moving on.


In [ ]:
print("=== CLEANING SUMMARY ===")
print(f"Device                         : {stats['device']}")
print(f"Chunks processed               : {stats['chunks_processed']:,}")
print(f"Rows read                      : {stats['rows_read']:,}")
print(f"Rows retained                  : {stats['rows_written']:,}")
print(f"Rows dropped                   : {stats['rows_dropped_missing_or_non_numeric']:,}")
print(f"+Infinity values replaced      : {stats['positive_infinity_values']:,}")
print(f"-Infinity values replaced      : {stats['negative_infinity_values']:,}")

drop_pct = 100 * stats["rows_dropped_missing_or_non_numeric"] / max(stats["rows_read"], 1)
print(f"Percentage of rows dropped    : {drop_pct:.4f}%")

print("\n=== CLASS DISTRIBUTION AFTER CLEANING ===")
for label, count in sorted(stats["labels"].items()):
    pct = 100 * count / stats["rows_written"]
    print(f"{label:22s} : {count:>12,} ({pct:6.2f}%)")


---
## 2.4 Duplicate inspection (do not delete automatically)

Repeated feature vectors can occur in network traffic data. We inspect duplicates, but retain them for now because removing repeated observations can change the class distribution.

The following optional check counts duplicates **within each chunk only**. It is a lower bound: duplicates occurring in different chunks will not be counted. A full-dataset duplicate check can require substantial RAM.


In [ ]:
# Optional: count duplicate feature rows within each chunk.
# This does NOT remove duplicates and does NOT detect matches across chunks.
duplicate_count_within_chunks = 0
rows_checked = 0

for chunk in pd.read_csv(OUTPUT_PATH, chunksize=CHUNK_SIZE, low_memory=False):
    cols = [c for c in chunk.columns if c != "label"]
    duplicate_count_within_chunks += int(chunk.duplicated(subset=cols).sum())
    rows_checked += len(chunk)

print(f"Rows checked                         : {rows_checked:,}")
print(f"Duplicate feature rows within chunks : {duplicate_count_within_chunks:,}")
print("Note: duplicates across different chunks are not included.")
print("No rows have been removed based on duplicate status.")


---
## 2.5 Save the cleaning report

The cleaned CSV and JSON report are saved in the same `preprocessed` folder. Step 3 can use the cleaned CSV directly.


In [ ]:
stats["duplicate_feature_rows_within_chunks"] = duplicate_count_within_chunks
stats["duplicate_check_note"] = (
    "Within-chunk duplicate count only; cross-chunk duplicates are not counted. "
    "Duplicates were inspected but not removed."
)

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2)

print("✓ Step 2 checkpoint saved!")
print("Cleaned dataset :", OUTPUT_PATH)
print("Cleaning report :", REPORT_PATH)
print(f"Cleaned file size: {OUTPUT_PATH.stat().st_size / (1024**2):.1f} MB")
print("\n🚀 Step 2 complete. Ready for Step 3: Prepare Labels.")
